In [ ]:
import os
import polars as pl
import pandas as pd
import numpy as np
import json
import time
from datetime import datetime, timedelta
from src.utils.helper import  convert_row, safe_get
from src.utils.config import setup_clickhouse_client
from src.utils.spark import get_spark
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, balanced_accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight

spark = get_spark()

ch_user = os.getenv('CLICKHOUSE_USER')
ch_password = os.getenv('CLICKHOUSE_PW')
ch_host = os.getenv('CLICKHOUSE_HOST')
client = setup_clickhouse_client(ch_user, ch_password, ch_host)

In [ ]:
# Fetch flights data
flights_query = "SELECT * FROM aviation_flights order by (dep_scheduled_time, dep_actual_time, arr_scheduled_time, arr_actual_time)"
flights_result = client.query(flights_query).result_rows
flights_columns = client.query(flights_query).column_names
flights_result = [convert_row(row) for row in flights_result]

# Create spark DataFrame
flights_pdf = pl.DataFrame(flights_result, schema=flights_columns, orient='row')

print(f"Total flight records retrieved: {len(flights_pdf)}")
flights_pdf.head(5)

In [ ]:
weather_query = "SELECT * FROM historical_weather_data order by date_observed"

weather_result = client.query(weather_query).result_rows
weather_result = [convert_row(row) for row in weather_result] 
weather_columns = client.query(weather_query).column_names

weather_pdf = pl.DataFrame(weather_result, schema=weather_columns, orient='row')

print(f"Total weather records retrieved: {len(weather_pdf)}")
weather_pdf.head(5)

In [ ]:
print('Prepared flights and weather data for merging.')

# Filter out cancelled and unknown status flights
valid_statuses = ['cancelled', 'unknown']
flights_filtered = flights_pdf.filter(~pl.col('status').is_in(valid_statuses)) \
                    .unique(subset=['flight_type', 'status', 'iata_number', 'airline_name', 
                                    'dep_scheduled_time', 'arr_scheduled_time', 'dep_actual_time', 'arr_actual_time'],
                                    keep='first')

print(f"Flight records after filtering: {len(flights_filtered)}")

In [ ]:
# Create hour columns for both departure and arrival times
flights_filtered = flights_filtered.with_columns(
    pl.col('dep_scheduled_time').str.to_datetime().dt.truncate('1h').alias('dep_hour'),
    pl.col('arr_scheduled_time').str.to_datetime().dt.truncate('1h').alias('arr_hour')
)

# Split into departure and arrival flights for conditional merging 
departure_flights = flights_filtered.filter(pl.col('flight_type') == 'departure')
arrival_flights = flights_filtered.filter(pl.col('flight_type') == 'arrival')

print(f"\nDeparture flights: {len(departure_flights)}, Arrival flights: {len(arrival_flights)}")
departure_flights.head(3)
arrival_flights.head(3)

In [ ]:
weather_pdf = weather_pdf.with_columns(
    pl.col('date_observed').str.to_datetime().dt.truncate('1h').alias('weather_hour') 
)

print(f"Weather records by hour: {len(weather_pdf)}")

weather_pdf.head(3)

In [ ]:
# Merge departure flights with weather
# Departure: join dep_hour with weather_hour
if len(departure_flights) > 0:
    merged_departures = departure_flights.join(
        weather_pdf,
        how='left',
        left_on='dep_hour',
        right_on='weather_hour',
        maintain_order='left'
    )

    merged_departures= merged_departures.drop(['arr_scheduled_time', 'arr_actual_time', 'arr_delay_mins']) \
        .sort(by=['dep_scheduled_time', 'dep_actual_time'], nulls_last=True) \
        .rename({'dep_scheduled_time': 'scheduled_time', 'dep_actual_time': 'actual_time', 'dep_delay_mins': 'delay_mins'})

    print(f"Merged departure flights: {len(merged_departures)}")
    merged_departures.head(3)

In [ ]:
# Merge arrival flights with weather
# Arrival: join arr_hour with weather_hour (same hour)
if len(arrival_flights) > 0:
    merged_arrivals = arrival_flights.join(
        weather_pdf,
        how='left',
        left_on='arr_hour',
        right_on='weather_hour',
        maintain_order='left'
    )
    
    merged_arrivals= merged_arrivals.drop(['dep_scheduled_time', 'dep_actual_time', 'dep_delay_mins']) \
    .sort(by=['arr_scheduled_time', 'arr_actual_time'], nulls_last=True) \
    .rename({'arr_scheduled_time': 'scheduled_time', 'arr_actual_time': 'actual_time', 'arr_delay_mins': 'delay_mins'})

    print(f"Merged arrival flights: {len(merged_arrivals)}")
    merged_arrivals.head(3)

In [ ]:
# Combine both datasets
if len(merged_departures) > 0 and len(merged_arrivals) > 0:
    combined_flights = pl.concat([merged_departures, merged_arrivals], how='vertical') \
            .sort(by=['scheduled_time', 'actual_time'], nulls_last=True)
    
    combined_flights = combined_flights.drop(['id', 'insert_time', 'iata_number', 'airline_name', 'codeshared_airline', 'codeshared_flight_number'])
else:
    combined_flights = pl.DataFrame()

print(f"\n Combined flights with weather data: {len(combined_flights)} rows")
combined_flights.head(3)

In [ ]:
# Machine Learning and prediction below
#
#

# Select features and target
feature_list = ["current_temp", "feels_like_temp", "pressure_hPa", "humidity_pct", "wind_speed_ms", "wind_deg", "cloudiness_pct", "rain_1h", "weather_main"]
features_var = combined_flights.select(feature_list)
independent_var = combined_flights["delay_mins"]

In [ ]:
y_statement = independent_var.to_numpy()
y_binary = (y_statement >= 15).astype(int)
class_counts = np.bincount(y_binary)

print(f"Delays >= 15 min: {class_counts[0]}, otherwise: {class_counts[1]}")

features_var = features_var.to_dummies(columns=["weather_main"])
X = combined_flights.select(features_var).to_numpy()
X = np.nan_to_num(X, nan=0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42
)

sample_weights = compute_sample_weight('balanced', y_train)

In [ ]:
#  Test binary class problem with random forest decision trees
#
#

rf_results = []

rf_model = RandomForestClassifier(n_estimators=250, random_state=42)
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='accuracy')

# Train on full train set for final evaluation
rf_model.fit(X_train, y_train, sample_weight=sample_weights)
y_pred = rf_model.predict(X_test)
print(y_pred[:10])
print(y_test[:10])

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
bal_acc_score=balanced_accuracy_score(y_test, y_pred)
class_rpt = classification_report(y_test, y_pred, target_names=['delayed', 'not delayed'], digits=4)

rf_results.append({
    'cv_mean_accuracy': cv_scores.mean(),
    'cv_std': cv_scores.std(),
    'test_accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1,
    'bal_acc_score': bal_acc_score,
    'class_rpt': class_rpt
})

results_rf = pd.DataFrame(rf_results)
print(results_rf[['cv_mean_accuracy', 'cv_std', 'test_accuracy', 'precision', 'recall', 'f1_score']].to_string())

print("\n\nSummary of Random Forest Algorithm for delay prediction:")
best = results_rf.iloc[0]
print(f"Cross-Val Accuracy (mean±std): {best['cv_mean_accuracy']:.2%} ± {best['cv_std']:.2%}")
print(f"Test Accuracy: {best['test_accuracy']:.2%}")
print(f'Balanced accuracy score: {balanced_accuracy_score(y_test, y_pred):.2%}')
print(f"Precision: {best['precision']:.2%}")
print(f"Recall: {best['recall']:.2%}")
print(f"f1_score: {best['f1_score']:.2%}")
print(f"Classification report: {class_rpt}")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
#  Test binary class problem with gradient boost
#
#

gb_results = []

# Model
gb_model = HistGradientBoostingClassifier(
    max_iter=500,
    learning_rate=0.05,
    max_depth=10,
    min_samples_leaf= 10,
    random_state=42,
    warm_start=True
)

cv_scores = cross_val_score(gb_model, X_train, y_train, cv=5, scoring='accuracy')

# Train on full train set for final evaluation
gb_model.fit(X_train, y_train, sample_weights)
y_pred = gb_model.predict(X_test)
print(y_pred[:10])
print(y_test[:10])

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
bal_acc_score=balanced_accuracy_score(y_test, y_pred)
class_rpt = classification_report(y_test, y_pred, target_names=['delayed', 'not delayed'], digits=4)

# Metrics
gb_results.append({
    'cv_mean_accuracy': cv_scores.mean(),
    'cv_std': cv_scores.std(),
    'test_accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1,
    'bal_acc_score': bal_acc_score,
    'class_rpt': class_rpt
})

results_gb = pd.DataFrame(gb_results)
print(results_rf[['cv_mean_accuracy', 'cv_std', 'test_accuracy', 'precision', 'recall', 'f1_score']].to_string())

print('\n\n Summary of Random Forest Algorithm for delay prediction:')
best = results_gb.iloc[0]

print(f"Cross-Val Accuracy (mean±std): {best['cv_mean_accuracy']:.2%} ± {best['cv_std']:.2%}")
print(f"Test Accuracy: {best['test_accuracy']:.2%}")
print(f'Balanced accuracy score: {balanced_accuracy_score(y_test, y_pred):.2%}')
print(f"Precision: {best['precision']:.2%}")
print(f"Recall: {best['recall']:.2%}")
print(f"f1_score: {best['f1_score']:.2%}")
print(f"Classification report: {class_rpt}")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
# ignore below, raw prediction testing 
#
#

# with open('raw/forecasted_weather_0413_to_0417.json', 'r', encoding='utf-8') as file:
#     data = json.load(file)

# col_name = ["date_observed", "current_temp", "feels_like_temp", "pressure_hPa", "humidity_pct", "min_temp", "max_temp", "wind_speed_ms", 
#            "wind_deg", "cloudiness_pct", "rain_1h", "weather_main", "weather_desc", "delay_expected"]
# final_list = []
# for item in data['list']:
#     time_in_utc = safe_get(item, 'dt_txt', default_value="2099-12-31T00:00:00.000")
#     time_in_sg = datetime.strptime(time_in_utc, '%Y-%m-%d %H:%M:%S') + timedelta(hours=8)
#     row_item = {
#         "date_observed": time_in_sg,
#         "current_temp": safe_get(item, 'main', 'temp', default_value= 999.99),
#         "feels_like_temp": safe_get(item, "main", "feels_like", default_value=999.99),
#         "pressure_hPa": safe_get(item, "main", "pressure", default_value=-100),
#         "humidity_pct": safe_get(item, "main", "humidity", default_value=-100),
#         "min_temp": safe_get(item, "main", "temp_min", default_value=-999.99),
#         "max_temp": safe_get(item, "main", "temp_max", default_value=999.99),
#         "wind_speed_ms": safe_get(item, "wind", "speed", default_value=0.00),
#         "wind_deg": safe_get(item, "wind", "deg", default_value=0), 
#         "cloudiness_pct": safe_get(item, "clouds", "all", default_value=-100),
#         "rain_1h": safe_get(item, "rain", "1h", default_value=0.00),
#         # "weather_main": , 
#         # "weather_desc": , 
#         # "delay_expected": 
#     }

#     # For each weather item, create a separate row with weather_main and weather_desc
#     weather_items = item.get("weather", [])
#     if weather_items:
#         for w in weather_items:
#             row_item = row_item.copy()
#             row_item["weather_main"] = w.get("main", "")
#             row_item["weather_desc"] = w.get("description", "")
#             final_list.append(row_item)
#     else:
#         row_item = row_item.copy()
#         row_item["weather_main"] = ""
#         row_item["weather_desc"] = ""
#         final_list.append(row_item)

# print(len(final_list))
# print(final_list[0])

In [ ]:
# # random forest
# final_list_df = pl.from_dicts(final_list)
# final_list_df = final_list_df.select(feature_list)
# final_list_df = final_list_df.to_dummies(columns=["weather_main"])
# final_list_df = final_list_df.with_columns(weather_main_Haze= pl.lit(0), weather_main_Mist= pl.lit(0), weather_main_Thunderstorm= pl.lit(0), weather_main_null= pl.lit(0))

# X_pred = final_list_df.select(final_list_df).to_numpy()
# X_pred = np.nan_to_num(X_pred, nan=0)

# y_pred = rf_model.predict(X_pred)
# print(y_pred)

In [ ]:
# # gradient boost
# y_pred_2 = gb_model.predict(X_pred)
# print(y_pred_2)

In [ ]:
# # final results
# y_pred_list = y_pred.tolist()

# for i in range(len(final_list)):
#     final_list[i]['delay_expected'] = y_pred_list[i]
# print(final_list[0])

# for i in range(0, len(final_list), 50):
#     end = min(i+50, len(final_list))
#     batch = final_list[i:end]
#     batch_data = [
#         [record.get(col, None) for col in col_name]
#         for record in batch
#     ]

#     client.insert('predict_flight_delays', batch_data, column_names=col_name)
#     print(f"Inserted {len(batch)} records into ClickHouse.")
#     time.sleep(2)